# Klasifikasi Status Kelangsungan Hidup Pasien Operasi pada Dataset Haberman

**Kelompok 4**
- Mohammad Zaydan Alrafi (10303240).
- Mohammad Bagus Satrio (103032400099).

**Algoritma yang di gunakan:** Decision Tree (Rule-based) vs K-Nearest Neighbors (Distance-based).

## 1. Pendahuluan & Pemaparan Data
Laporan kali ini, kelompok kami akan menganalisis dataset Haberman's Survival.
Dataset ini di ambil dari studi yang dilakukan di University of Chicago's Billings Hospital pada tahun 1958 dan 1970 untuk pasien yang menjalani operasi kanker payudara.

Tujuan utama kami adalah mencoba untuk memprediksi status kelangsungan hidup pasien (apakah bertahan hidup lebih dari 5 tahun atau kurang) berdasarkan tiga fitur yaitu:
 Usia pasien saat operasi, Tahun operasi, dan Jumlah *positive axillary nodes* yang terdeteksi.

Mari kita *load* datanya dan melihat bagaimana bentuk dan distribusi kelasnya.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('haberman.csv')
df.columns = ['age', 'op_Year', 'axil_nodes', 'surv_status']

print("tampilkan baris pertama:")
display(df.head())

print("\nstatus kelangsungan hidup:")
print("1 = bertahan hidup 5 tahun atau lebih\n2 = meninggal dalam 5 tahun")
print(df['surv_status'].value_counts())


plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='surv_status', palette='Set2')
plt.title('distribusi kelas (survival status)')
plt.xlabel('Status (1 = >= 5 tahun, 2 = < 5 tahun)')
plt.ylabel('jumlah Pasien')
plt.show()

ModuleNotFoundError: No module named 'pandas'

**Observasi Data:**
Dari data tersebut, terlihat kalau data kita imbalanced atau timpang, pasien yang bertahan hidup lebih dari 5 tahun (Kelas 1) jauh lebih banyak dibandingkan yang meninggal dalam 5 tahun (Kelas 2).
Jika kita langsung melatih model pakai data ini, modelnya nanti akan cenderung nebak Kelas 1 terus karena probabilitas awalnya sudah tinggi atau bias ke mayoritas.
Mmasalah ini akan menjadi pelajaran selanjutnya supaya modelnya lebih fair dan pintar mengenali Kelas 2.

## 2. Pra-Pemrosesan Data
Sebelum kita ke modeling. kita ubah dulu datanya menjadi data latih (train) dan data uji (test). 
Setelah itu, kita perlu melakukan normalisasi atau feature scaling menggunakan standard scaler.

**Kenapa Scaling ini penting banget buat KNN?**
KNN itu distance-based, artinya ia menghitung jarak antar titik data (misalkan menggunakan jarak Euclidean).

Kalau ada fitur yang rentang nilainya beda jauh (misal umur 30-90, sedangkan tahun operasinya cuma 58-70), fitur yang nilainya besar akan mendominasi perhitungan jarak. Yang mengakibatkan, KNN menganggap fitur yang rentangnya besar itu lebih penting, padahal belum tentu benar, dengan menggunakan standard scaler, kita menyamakan skala fitur (rata-rata jadi 0, standar deviasi 1). 

Kalau Decision Tree rule-based, scaling. Sebenarnya tidak akan berpengaruh. Decision Tree cuma bikin aturan if-else (misal: if Age > 50) tanpa mempedulikan skala antar fiturnya. Supaya gampang membandingkan kedua model dengan data yang sama, kita menerapkan scaling ke data tersebut.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop('surv_status', axis=1)
y = df['surv_status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Dimensi X_train sebelum scaling:", X_train.shape)
print("Dimensi X_test:", X_test.shape)

Dimensi X_train sebelum scaling: (244, 3)
Dimensi X_test: (62, 3)


## 3. Penanganan Imbalanced Data (SMOTE)
Seperti yang kita temukan pas EDA tadi, data kita sangat timpang. Untuk ngatasin ini, kita pakai teknik SMOTE (Synthetic Minority Over-sampling Technique). 

Cara kerjanya yaitu: menduplikasi data minoritas (yang bisa bikin overfitting), SMOTE ini bikin data sintetis baru berdasarkan tetangga-tetangga terdekat dari data minoritas yang ada. Jadi data training kita bakal seimbang antara Kelas 1 dan Kelas 2. SMOTE hanya diaplikasikan ke data training saja Kita jangan manipulasi data testing biar ujiannya tetap nyata.

In [ ]:

##Inisialisasi SMOTE
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print("Distribusi kelas sebelum SMOTE:")
print(y_train.value_counts())

print("\nDistribusi kelas sesudah SMOTE:")
print(y_train_smote.value_counts())

Distribusi kelas sebelum SMOTE:
surv_status
1    179
2     65
Name: count, dtype: int64

Distribusi kelas sesudah SMOTE:
surv_status
2    179
1    179
Name: count, dtype: int64


## 4. Metode & Eksperimen
Kita akan coba melatih dua algoritma: KNN dan Decision Tree.

Supaya performa model lebih optimal, kita bakal menggunakan `GridSearchCV`. Ini berguna untuk eksperimen dan mencari *hyperparameter* terbaik dengan kombinasi *Cross-Validation*.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

knn_params = {'n_neighbors': range(1, 21), 'weights': ['uniform', 'distance']}
knn = KNeighborsClassifier()

print("Sedang mencari parameter terbaik untuk KNN...")
knn_grid = GridSearchCV(knn, knn_params, cv=5, scoring='recall') 
knn_grid.fit(X_train_smote, y_train_smote)

print("Parameter KNN terbaik:", knn_grid.best_params_)

dt_params = {'max_depth': [3, 5, 7, 10, None], 'min_samples_split': [2, 5, 10]}
dt = DecisionTreeClassifier(random_state=42)

print("\nSedang mencari parameter terbaik untuk Decision Tree...")
dt_grid = GridSearchCV(dt, dt_params, cv=5, scoring='recall')
dt_grid.fit(X_train_smote, y_train_smote)

print("Parameter Decision Tree terbaik:", dt_grid.best_params_)

best_knn = knn_grid.best_estimator_
best_dt = dt_grid.best_estimator_